# MBN GUIDE | reproducible portfolio evidence

This notebook is an **audit-oriented portfolio walkthrough**. It does not crawl MBN, call a provider, load an embedding model, or generate a release. Instead, it independently reads the pinned local data-engine artifacts, verifies the M14R1 payload manifest, and reports the separate one-year backfill checkpoint.

Required local input: an authorized checkout of `mbN_GUIDE` pointed to by `MBN_GUIDE_PY_ROOT`. No raw bodies, vectors, model cache, or secrets are copied into this portfolio repository.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import pandas as pd

ROOT = Path(os.environ.get('MBN_GUIDE_PY_ROOT', '../mbN_GUIDE_PY/mbN_GUIDE')).expanduser().resolve()
assert ROOT.exists(), f'Set MBN_GUIDE_PY_ROOT to the mbN_GUIDE checkout: {ROOT}'
RELEASE_ID = 'mbn-guide-701011c9608e4524'
RELEASE = ROOT / 'data/90_exports/frontend' / RELEASE_ID
M14R1_GATE = ROOT / 'data/80_quality/mbn/m14r1/run_20260808_m14r1_validation/m14r1_final_gate.json'
M15_STATUS = ROOT / 'data/80_quality/mbn/m15/run_20260808_m15b_deterministic/m15b_status.json'
assert RELEASE.exists() and M14R1_GATE.exists() and M15_STATUS.exists(), 'Required artifact paths are missing from this checkout.'
print('Source checkout:', ROOT)
print('Release:', RELEASE_ID)

## 1. Validate immutable release payloads

The release manifest is re-read and every payload byte size and SHA-256 is recomputed. A mismatch is an audit failure; this notebook never rewrites the bundle.

In [ ]:
manifest = json.loads((RELEASE / 'manifest.json').read_text(encoding='utf-8'))
checks = []
for item in manifest['files']:
    path = RELEASE / item['path']
    payload = path.read_bytes()
    checks.append({
        'path': item['path'],
        'rows_manifest': item.get('rows'),
        'bytes_manifest': item['bytes'],
        'bytes_actual': len(payload),
        'sha_match': hashlib.sha256(payload).hexdigest() == item['sha256'],
    })
release_checks = pd.DataFrame(checks)
assert (release_checks.bytes_manifest == release_checks.bytes_actual).all()
assert release_checks.sha_match.all()
release_checks

In [ ]:
gate = json.loads(M14R1_GATE.read_text(encoding='utf-8'))
critical = [
    'jsonParseErrors', 'duplicateIds', 'duplicateRecommendations',
    'duplicateRelatedArticles', 'brokenFK', 'relatedCountViolation',
    'relatedRankGap', 'relatedNaNInf', 'relatedReasonMissing',
    'manifestMismatch', 'hashMismatch', 'byteMismatch', 'secretLeak',
]
assert gate['promotionVerdict'] == 'FRONTEND_READY'
assert all(gate.get(key, 0) == 0 for key in critical)
pd.DataFrame([{'releaseId': gate['releaseId'], 'validationStatus': gate['validationStatus'], 'promotionVerdict': gate['promotionVerdict'], **{key: gate.get(key) for key in critical}}])

## 2. Read the frontend-safe projection

These counts describe the product-safe JSON projection, not the full internal candidate graph. In particular, related articles are exposed separately from compact place recommendations.

In [ ]:
summary = {
    'articles': len(json.loads((RELEASE / 'articles.json').read_text(encoding='utf-8'))),
    'places': len(json.loads((RELEASE / 'places.json').read_text(encoding='utf-8'))),
    'stories': len(json.loads((RELEASE / 'stories.json').read_text(encoding='utf-8'))),
    'recommendations': len(json.loads((RELEASE / 'recommendations.json').read_text(encoding='utf-8'))),
    'related_articles': len(json.loads((RELEASE / 'related_articles.json').read_text(encoding='utf-8'))),
    'taxonomy_categories': len(json.loads((RELEASE / 'taxonomy.json').read_text(encoding='utf-8'))),
}
pd.DataFrame([summary])

## 3. Inspect the non-release one-year checkpoint

M15 is intentionally reported separately. It is a deterministic backfill and incremental-embedding checkpoint, **not** a replacement for the frozen release. The code below makes that boundary explicit.

In [ ]:
m15 = json.loads(M15_STATUS.read_text(encoding='utf-8'))
pd.DataFrame([{
    'm15a_status': m15['m15a']['status'],
    'combined_index': m15['m15a']['combinedIndex'],
    'combined_eligible': m15['m15a']['combinedEligible'],
    'body_blocks': m15['m15a']['bodyBlocks'],
    'requires_independent_llm': m15['delta']['requiresLLM'],
    'combined_title_vectors': m15['embedding']['combinedTitleRows'],
    'combined_chunk_vectors': m15['embedding']['combinedChunkRows'],
    'pre_llm_verdict': m15['temporal']['verdict'],
    'one_year_data_ready': m15['oneYearDataReady'],
}])

## Method map and interpretation

- **M7:** provider resolution produces canonical places/events only after evidence and map-safety gates.
- **M8:** a pinned BGE-M3 vector space is numerically audited before use.
- **M9:** body-first exact cosine matrices are aggregated at article level; Top-K is a relation measurement, not a recommendation.
- **M10–M12:** editorial weak labels, retrieval evidence, ranking features, and deterministic reasons remain distinct.
- **M13–M14R1:** only contract-compatible objects are projected. Unsupported Event/Story/article-context surfaces are recorded as gaps instead of coerced.

A successful notebook run establishes artifact integrity for the referenced checkout. It does not turn weak labels into human gold, or a pre-LLM backfill checkpoint into a release.